# 搭建一个RAG--文档处理

- 加载：首先我们需要加载数据，通过文档加载器完成。
- 拆分：文本拆分器将大块数据拆分Documents成更小的块。这对于索引数据和将数据传入模型都非常有用，因为大块数据更难搜索，而且无法容纳在模型有限的上下文窗口中。
- 存储：我们需要一个地方来存储和索引我们的分割数据，以便日后进行搜索。这通常使用VectorStore和Embeddings模型来实现。

In [2]:
import os
from langchain.chat_models import ChatOpenAI
from langchain.schema import AIMessage, HumanMessage, SystemMessage, ChatMessage
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredWordDocumentLoader, CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import Dict, List, Optional, Tuple, Union

import warnings

warnings.filterwarnings("ignore")

## 文档加载器
- langchain_community.document_loaders 模块提供了统一的接口
能将不同格式的数据源转换为标准的 Document 对象（包含文本内容 page_content 和元数据 metadata），为后续处理（如文本分割、向量化）奠定基础。下面这个表格汇总了几种常见文档加载器及其典型使用场景。

| 加载器类型 | 主要适用场景 | 关键特点说明 |
| :--- | :--- | :--- |
| **`TextLoader`** | 加载纯文本文件（如 `.txt`, `.md`） | 处理简单，适用于无需复杂解析的文本。 |
| **`CSVLoader`** | 加载结构化表格数据（CSV文件） | 默认将每一行转换为一个 Document，可自定义分隔符等参数，并指定标识源的列 (`source_column`)。 |
| **`PyPDFLoader`** | 提取PDF文件中的文本内容 | 按页分割文档，每页成为一个 Document 对象。 |
| **`DirectoryLoader`** | 批量加载目录下的多个文件 | 通过 `glob` 模式匹配文件，支持多线程 (`use_multithreading=True`) 和进度条显示 (`show_progress=True`)。 |
| **`WebBaseLoader`** | 抓取网页文本内容 | 适用于需要从互联网获取实时信息的场景。 |
| **`JSONLoader`** | 解析JSON或JSON Lines文件 | 使用 `jq_schema` 灵活指定要提取的字段。 |
| **`UnstructuredFileLoader`** | 处理多种文件格式（如PPT、Word） | 依赖 `unstructured` 库，能处理非纯文本格式的文档。 |

- 一些技巧
1. 批量操作与性能提升：使用 DirectoryLoader 可以一次性加载整个目录的文件。通过设置 use_multithreading=True 可以启用多线程，显著提升大量文件的加载速度；设置 show_progress=True 则可以显示加载进度条。

2. 精细化控制数据来源：对于 CSVLoader，可以使用 source_column 参数指定表中的某一列作为文档的源信息（metadata 中的 source 字段）。这在后续检索增强生成（RAG）等应用中，有助于追踪信息的原始出处。

3. 理解不同加载器的输出差异：同样是处理CSV文件，CSVLoader 会将每一行作为一个独立的 Document；而 UnstructuredCSVLoader 则可能将整个表格作为一个元素加载。你需要根据后续处理的需求（例如，是按行检索还是整体分析表格）来选择合适的加载器。

In [3]:
documents = CSVLoader(file_path="../data/d1/movie.csv", source_column='国家', encoding="utf-8").load()
documents

[Document(metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='编号: 31\n电影名称: 天堂电影院\n导演: 朱塞佩·托纳多雷 Giuseppe Tornatore\n主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情\n评分人数: 397297'),
 Document(metadata={'source': '日本', 'row': 2}, page_content='编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本\n分类: 动画 剧情 战争\n评分人数: 257156'),
 Document(metadata={'source': '美国', 'row': 3}, page_content='编号: 151\n电影名称: 雨人\n导演: 巴瑞·莱文森 Barry Levinson\n主演: 奥黛丽·塔图 Audrey Tau...\n年份: 1988\n国家: 美国\n分类: 剧情\n评分人数: 255152'),
 Document(metadata={'source': '法国 美国 意大利', 'row': 4}, page_content='编号: 245\n电影名称: 碧海蓝天\n导演: Luc Besson\n主演: 汤姆·汉克斯 Tom Hanks...\n年份: 1988\n国家: 法国 美国 意大利\n分类: 剧情 爱情\n评分人数: 120197')]

## 1.文档分割

- langchain关于文档分割，提供了两个接口 CharacterTextSplitter &  RecursiveCharacterTextSplitter
- 不同的接口实现了不同的切割方式

### 基于长度的分割类型：CharacterTextSplitter
- token based：根据模型计算的token数量拆分文本，在使用语言模型时很有用。
- character based： 根据字符数拆分文本，这可以使不同类型的文本更加一致。

In [4]:
documents[0]

Document(metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293')

In [5]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=2, chunk_overlap=1)
texts = text_splitter.split_text(documents[0].page_content)  # 接受str
len(texts), texts

(1,
 ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'])

### 基于文本结构哦的分割类型：RecursiveCharacterTextSplitter
- 文本自然地被组织成段落、句子和单词等层级单元。利用这种固有结构（如分隔符，长度等）来指导我们的拆分策略。

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=40, chunk_overlap=20, separators=["\n\n", "\n", "。", "，", " ", ""])
docs = text_splitter.split_documents(documents)
docs

[Document(metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险'),
 Document(metadata={'source': '日本', 'row': 0}, page_content='国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='编号: 31\n电影名称: 天堂电影院'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='导演: 朱塞佩·托纳多雷 Giuseppe Tornatore'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情'),
 Document(metadata={'source': '意大利 法国', 'row': 1}, page_content='分类: 剧情 爱情\n评分人数: 397297'),
 Document(metadata={'source': '日本', 'row': 2}, page_content='编号: 140\n电影名称: 萤火虫之墓'),
 Document(metadata={'source': '

# LLM & Embedding 模型加载

In [7]:
from config import config
from llm_utils import SiliconFlowEmbedding, SiliconFlowChat

In [8]:
# 测试嵌入模型
emb_client = SiliconFlowEmbedding(api_key=config.llm_api_key,
                                           emb_model= config.emb_model,
                                           emb_api_url=config.emb_api_base)

test_texts = ["这是一个测试文档", "这是另一个测试文档"]
embeddings = emb_client.embed_documents(test_texts)
print(f"生成 {len(embeddings)} 个嵌入向量，每个维度为 {len(embeddings[0])}")

# 测试对话功能
chat_client = SiliconFlowChat(api_key=config.llm_api_key,
                                           chat_model= config.llm_model,
                                           chat_api_url= config.llm_api_base
                              )
messages = [
    {"role": "system", "content": "你是一个专业的AI助手。"},
    {"role": "user", "content": "讲一个冷效果"}
]

response = chat_client.chat_completion(messages)
if response:
    print("AI回复:", response)

生成 2 个嵌入向量，每个维度为 1024
AI回复: 好的，这里有一个冷笑话，希望能给你带来一丝清凉的感觉：

为什么电脑永远不会感冒？

因为它有“Windows”（窗户），但它是关闭的（关闭的窗户不容易让冷风或病毒进入）。

希望这个笑话能让你会心一笑，感受到一丝轻松和愉快。


## 2.向量库加载

In [9]:
vector_save_path = 'VectorStores/test_storage'
if not os.path.exists(vector_save_path):
    vector = FAISS.from_documents(documents, emb_client)
    vector.save_local(vector_save_path)
else:
    vector = FAISS.load_local(folder_path=vector_save_path, embeddings=emb_client,
                              allow_dangerous_deserialization=True)

## 3.LLM Chat Model定义

In [10]:
PROMPT_TEMPLATE = dict(
    RAG_PROMPT_TEMPALTE="""结合以上下文来回答用户的问题。
        问题: {question}
        可参考的上下文：
        ···
        {context}
        ···
        如果给定的上下文无法让你做出回答，请回答数据库中没有这个内容，不要臆想推测，请使用中文回答。
        回答:""",
)

## 4.RAG问答

In [11]:
query = '推荐一部宫崎骏的电影'
contents = vector.similarity_search_with_score(query, k=3)
print(f"检索结果： {contents}")
context = [c[0].page_content for c in contents]
print(f"检索结果文本： {context}")

检索结果： [(Document(id='4fe0e5fd-70f2-46c0-b7ba-e633cc271be5', metadata={'source': '日本', 'row': 0}, page_content='编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293'), np.float32(1.0679572)), (Document(id='d1d80a08-00e9-4edb-8367-54930f48a2da', metadata={'source': '日本', 'row': 2}, page_content='编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本\n分类: 动画 剧情 战争\n评分人数: 257156'), np.float32(1.3351365)), (Document(id='ca566466-dd0b-4695-abda-046a6f176de1', metadata={'source': '意大利 法国', 'row': 1}, page_content='编号: 31\n电影名称: 天堂电影院\n导演: 朱塞佩·托纳多雷 Giuseppe Tornatore\n主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情\n评分人数: 397297'), np.float32(1.4373972))]
检索结果文本： ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293', '编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本

In [12]:
content = PROMPT_TEMPLATE['RAG_PROMPT_TEMPALTE'].format(question=query, context=context)
print(content)

messages = [
    {"role": "system", "content": "你是一个专业的AI助手。"},
    {"role": "user", "content": content}
]
response = chat_client.chat_completion(messages)
response

结合以上下文来回答用户的问题。
        问题: 推荐一部宫崎骏的电影
        可参考的上下文：
        ···
        ['编号: 18\n电影名称: 龙猫\n导演: 宫崎骏 Hayao Miyazaki\n主演: 日高法子 Noriko Hidaka / 坂本千夏 Ch...\n年份: 1988\n国家: 日本\n分类: 动画 奇幻 冒险\n评分人数: 678293', '编号: 140\n电影名称: 萤火虫之墓\n导演: 高畑勋 Isao Takahata\n主演: 法拉赫阿米尔·哈什米安 Amir Fa...\n年份: 1988\n国家: 日本\n分类: 动画 剧情 战争\n评分人数: 257156', '编号: 31\n电影名称: 天堂电影院\n导演: 朱塞佩·托纳多雷 Giuseppe Tornatore\n主演: 莱昂纳多·迪卡普里奥 Leonardo...\n年份: 1988\n国家: 意大利 法国\n分类: 剧情 爱情\n评分人数: 397297']
        ···
        如果给定的上下文无法让你做出回答，请回答数据库中没有这个内容，不要臆想推测，请使用中文回答。
        回答:


'推荐您观看宫崎骏导演的电影《龙猫》。这部电影在1988年于日本上映，是一部动画奇幻冒险电影。'